In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Quantum Algorithm Simulation 101 — Simulation: Choosing the Right Simulator
$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}$

---

Quantum algorithm simulation spans several complementary techniques. This notebook focuses on how state vector, tensor network, matrix product state, Pauli propagation, and stabilizer simulators differ in memory cost, observable access, and the kinds of circuits they handle well.

**What You Will Do:**
* Build small NumPy and CUDA-Q examples for state vector, tensor network, and matrix product state simulation
* Compare when state vector, tensor network, Pauli propagation, and stabilizer methods are the right fit
* Analyze how endianness, contraction order, bond dimension, and branching affect simulator performance

**Prerequisites:**
* Python and Jupyter notebook familiarity
* Basic knowledge of quantum computing (qubits, gates, circuits, bra-ket notation, measurement)
* Familiarity with expectation values and Pauli operators is helpful

**Key Terminology:**
* State vector (SV)
* Tensor network (TN)
* Contraction path
* Matrix product state (MPS)
* Bond dimension
* Clifford gate
* Stabilizer
* Pauli propagation

**CUDA-Q Syntax:**
* [`@cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines a quantum kernel function
* [`cudaq.sample`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample) — samples measurement outcomes from a kernel
* [`cudaq.observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe) — computes expectation values of spin operators
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects a simulation backend such as state vector, tensor network, or STIM

**Solutions:** [`solutions/01_simulation101_solutions.ipynb`](solutions/01_simulation101_solutions.ipynb)


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 12px 15px 12px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900;">&#9889; GPU Required:</span>** This notebook requires a GPU.

</div>


In [22]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install CUDA-Q.
## Run this notebook in a GPU runtime.

!pip install cudaq cudaq-qec -q
#
#!wget -q https://github.com/nvidia/cuda-q-academic/archive/refs/heads/main.zip
#!unzip -q main.zip
#!mv cuda-q-academic-main/simulation/images ./images


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.9/128.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.7/407.7 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).


## State Vector Simulation

State vector (SV) simulation is the most common technique for algorithm simulation and probably the technique most users are familiar with. The method is also the most straightforward. For an $N$ qubit state, the corresponding $2^N$ element state vector is stored in memory. Gates are applied as matrix operations acting on this vector to produce the final state.

The advantage of this is that the entire state is known at all times. This allows us to perform any valid operation and save the entire state vector for other applications. This is closely related to its primary limitation, memory. Because the SV grows exponentially, SV simulation is limited to about 50 qubits. To produce simulations in the low 50s of qubits, entire supercomputers are necessary to have sufficient memory. This is unsurprising, as manipulation of an exponentially large Hilbert space is one of the reasons quantum computing could be advantageous at all.

In this notebook, we will use the basis ordering $\ket{q_0 q_1 q_2}$, so the leftmost factor in a Kronecker product acts on $q_0$. Being explicit about this endian convention is important because it determines how we build multi-qubit operators such as CNOT and how we interpret the printed amplitudes.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 1:</h3>
    <p style="font-size: 16px; color: #333;">
Use NumPy to prepare an identity gate, a Hadamard gate, and a CNOT gate. Build the state vector corresponding to $\ket{000}$ and prepare the $N=3$ GHZ state by performing the matrix multiplications like a state vector simulator would. Next, write a CUDA-Q kernel to prepare an arbitrary-sized GHZ state. On your device, how large of a state can you produce before a memory error occurs? Be explicit about the endian convention you assume, since it affects how you write the Kronecker products and multi-qubit matrices such as CNOT.
</div>


In [17]:
import numpy as np

# EXERCISE 1
# Basis ordering convention: |q0 q1 q2>, so the leftmost tensor factor acts on q0.
#
# ##TODO## Define the single-qubit gates I, H, and X.
I = np.array([
    [1,0],
    [0,1]
])

H = (1/np.sqrt(2)) * np.array([
    [1,1],
    [1,-1]
])

X = np.array([
    [0,1],
    [1,0]
])

# ##TODO## Define the 2-qubit CNOT matrix in the same basis convention.
CNOT = np.array([
    [1, 0, 0, 0],   # |00>
    [0, 1, 0, 0],   # |01>
    [0, 0, 0, 1],   # |10> -> |11>
    [0, 0, 1, 0]    # |11> -> |10>
])

# ##TODO## Initialize the |000> state vector.
psi = np.zeros(8, dtype=complex)
psi[0] = 1.0

# ##TODO## Build H_full, CNOT01_full, and CNOT12_full using np.kron.
H_full = np.kron(np.kron(H, I), I)
CNOT01_full = np.kron(CNOT, I)
CNOT12_full = np.kron(I, CNOT)

# ##TODO## Apply the operators in order to prepare the GHZ state.
psi = H_full @ psi
psi = CNOT01_full @ psi
psi = CNOT12_full @ psi
# ##TODO## Print the final state vector and the non-zero amplitudes.
print("Final state vector:")
print(psi)

print("\nNon-zero amplitudes:")

basis_states = [
    "|000>",
    "|001>",
    "|010>",
    "|011>",
    "|100>",
    "|101>",
    "|110>",
    "|111>",
]

for basis, amp in zip(basis_states, psi):
    if not np.isclose(amp, 0):
        print(f"{basis}: {amp}")


Final state vector:
[0.70710678+0.j 0.        +0.j 0.        +0.j 0.        +0.j
 0.        +0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]

Non-zero amplitudes:
|000>: (0.7071067811865475+0j)
|111>: (0.7071067811865475+0j)


### CUDA-Q Example


State vector simulations are awesome for algorithm development, especially as quantum computers are mostly within the range of tens of qubits. Users can generally run extremely deep circuits quickly, explore arbitrary noise models, and access any information about the state along the way. This makes SV simulation very useful for comparing noiseless results to QPUs and developing noise models to mimic QPUs with small numbers of qubits. The only reason you would not use SV simulation is when you want to scale beyond available memory. Most of the examples in the [CUDA-Q Application Hub](https://nvidia.github.io/cuda-quantum/latest/using/applications.html) use SV simulation.

Note: CUDA-Q's SV simulator is powered by NVIDIA's cuStateVec library, which is part of cuQuantum. This library is highly optimized for GPU performance and uses many tricks far beyond simply performing the matrix multiplications.

## Tensor Network Simulations

While the utility of SV simulation cannot be understated, it is critical to be able to simulate algorithms beyond $N=50$ (or less if you do not have a supercomputer available), especially as many QPUs already have more than 50 qubits. We now need to start exploring more sophisticated techniques that do not store the entire state, yet can sample from its probability amplitudes or produce observables.

Tensor network (TN) simulations are one such technique that can scale far beyond 50 qubits, even to the order of 10,000 qubits in certain cases. But you will see shortly there is still no free lunch, and this sort of scale can only be achieved for a narrow range of circuits. For many local circuits, the tensor network description itself grows roughly with the number of qubits times the circuit depth, while the actual runtime still depends strongly on the contraction path and the entanglement created by the circuit. This is why TN methods are especially attractive for shallow circuits.

A tensor is a general data structure that is said to have rank $n$. A rank-0, rank-1, and rank-2 tensor is a scalar, vector, and matrix, respectively, with higher-order tensors adding dimensions. The number of elements in each dimension of a tensor is called the extent. For fixed extent, the number of elements in a tensor grows exponentially with rank.

Tensors are often visually represented as nodes with lines coming from them, where a node by itself is a scalar and each line is a higher-order dimension.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/simulation/images/tensorranks.png?raw=1" alt="Examples of tensors with increasing rank" width="700">

An important operation for tensors is contraction, or summing over the shared indices of two or more tensors. Consider two matrices $A_{ij}$ and $B_{jk}$. If we multiply the two, we would sum over the common $j$ index to result in a new matrix $C_{ik}$. We are computing $C_{ik} = \sum_j A_{ij} B_{jk}$, which is often written in Einstein notation with the explicit sum dropped: $C_{ik} = A_{ij} B_{jk}$. These operations can be generalized to any-order tensors and represented graphically, where connected lines in a diagram are the indices summed over in the contraction. See the examples pictured below.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/simulation/images/tncontractions.png?raw=1" alt="Examples of tensor contractions between low-rank tensors" width="700">

Connecting many of these diagrams results in a tensor network. This is important because quantum circuits can be represented as tensor networks where each initial qubit is a vector, each single-qubit gate is a rank-2 tensor, and each two-qubit gate is a rank-4 tensor. If we just convert the circuit to a TN and contract, we recover our state vector.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 2:</h3>
    <p style="font-size: 16px; color: #333;">
Given the input tensors below, write an einsum to contract the whole network and recover the state vector. Note that the CNOT gate is converted into a rank-4 tensor for you.
</div>


In [20]:
import numpy as np

# EXERCISE 2
# We describe the circuit connectivity using Einstein summation conventions.
#
# Indices key:
# a: Input for q0
# b: Output of H (becomes control input for CNOT1)
# c: Input for q1 (target input for CNOT1)
# d: Final output for q0 (control output of CNOT1)
# e: Intermediate output for q1 (target output of CNOT1 -> control input for CNOT2)
# f: Input for q2 (target input for CNOT2)
# g: Final output for q1 (control output of CNOT2)
# h: Final output for q2 (target output of CNOT2)
#
# Endianness note: we list basis states as |q0 q1 q2>, so the leftmost tensor factor acts on q0.
#
# The Circuit:
# q0 --a--[H]--b--[ C ]--d (out q0)
#                  [ N ]
# q1 --c-----------[ O ]--e--[ C ]--g (out q1)
#                  [ T ]     [ N ]
# q2 --f---------------------[ O ]--h (out q2)
#                            [ T ]
#
# ##TODO## Define q0, q1, q2, H, and the rank-4 CNOT tensor.
# Single-qubit states
q0 = np.array([1, 0], dtype=complex)
q1 = np.array([1, 0], dtype=complex)
q2 = np.array([1, 0], dtype=complex)
# Hadamard gate
H = (1 / np.sqrt(2)) * np.array([
    [1,  1],
    [1, -1]
], dtype=complex)

# CNOT matrix in basis |control target> = |00>, |01>, |10>, |11>
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

# Convert CNOT into rank-4 tensor:
# CNOT[output_control, output_target, input_control, input_target]
CNOT4 = CNOT.reshape(2, 2, 2, 2)

# ##TODO## Use np.einsum to contract the network and produce a tensor with indices dgh.
psi_dgh = np.einsum(
    "ba,a,debc,c,ghef,f->dgh",
    H, q0,
    CNOT4, q1,
    CNOT4, q2
)

# ##TODO## Flatten the result into a state vector and verify it matches the GHZ state.
state = psi_dgh.reshape(-1)
print(state)

[0.70710678+0.j 0.        +0.j 0.        +0.j 0.        +0.j
 0.        +0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


Astute learners might wonder at this point, "how does this help us if we just recover the SV? Was storing the SV not the problem with SV simulation?" In practice, the contraction above would never be done. If we wanted to compute the expectation value of an observable, we would amend the observable to the TN, and then extend the network with the conjugate of $\ket{\psi}$, that is, produce a network that corresponds to $\bra{\psi} O \ket{\psi}$.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/simulation/images/tn_observe.png?raw=1" alt="Tensor network for an observable expectation value" width="700">

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 3:</h3>
    <p style="font-size: 16px; color: #333;">
Amend your code above and use it to compute the expectation value of $Z_0 Z_1$ and $Z_0 I_1$. To keep things organized, it might help to contract a tensor for the ket state and the bra state first, and then calculate the final contraction with the observable. Confirm that you agree with CUDA-Q's output.
</div>


In [23]:
import numpy as np
import cudaq
from cudaq import spin

# EXERCISE 3
# ##TODO## Define a CUDA-Q Bell-state kernel.
# ##TODO## Build the observables Z tensor Z and Z tensor I with spin operators.
# ##TODO## Compute those expectation values with cudaq.observe.
# ##TODO## Recreate the same calculation with an explicit tensor-network contraction.
# ##TODO## Confirm the NumPy and CUDA-Q results agree.

# -----------------------------
# 1. CUDA-Q Bell-state kernel
# -----------------------------

@cudaq.kernel
def bell_kernel():
    q = cudaq.qvector(2)

    h(q[0])
    x.ctrl(q[0], q[1])


# -----------------------------
# 2. CUDA-Q observables
# -----------------------------

Z0Z1 = spin.z(0) * spin.z(1)
Z0I1 = spin.z(0) * spin.i(1)


# -----------------------------
# 3. CUDA-Q expectation values
# -----------------------------

cudaq_Z0Z1 = cudaq.observe(bell_kernel, Z0Z1).expectation()
cudaq_Z0I1 = cudaq.observe(bell_kernel, Z0I1).expectation()

print("CUDA-Q <Z0 Z1> =", cudaq_Z0Z1)
print("CUDA-Q <Z0 I1> =", cudaq_Z0I1)


# -----------------------------
# 4. NumPy tensor-network version
# -----------------------------

q0 = np.array([1, 0], dtype=complex)
q1 = np.array([1, 0], dtype=complex)

H = (1 / np.sqrt(2)) * np.array([
    [1,  1],
    [1, -1]
], dtype=complex)

CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

# CNOT4[output_control, output_target, input_control, input_target]
CNOT4 = CNOT.reshape(2, 2, 2, 2)

# Build the ket state |psi>
#
# Circuit:
#
# q0: a -- H -- b -- CNOT -- d
# q1: c --------- CNOT -- e
#
# Final ket indices are d,e.
ket_de = np.einsum(
    "ba,a,c,debc->de",
    H, q0, q1, CNOT4
)

# Build the bra state <psi|
#
# Same tensor as ket, but complex conjugated.
bra_de = np.conjugate(ket_de)

# Pauli operators
Z = np.array([
    [1,  0],
    [0, -1]
], dtype=complex)

I = np.eye(2, dtype=complex)

# Observable tensors
Z0Z1_np = np.einsum("ij,kl->ikjl", Z, Z)
Z0I1_np = np.einsum("ij,kl->ikjl", Z, I)

# The observable tensors have indices:
#
# Z0Z1_np[d_out, e_out, d_in, e_in]
# Z0I1_np[d_out, e_out, d_in, e_in]
#
# Therefore:
#
# <psi|O|psi> =
# bra[d_out, e_out] O[d_out, e_out, d_in, e_in] ket[d_in, e_in]

numpy_Z0Z1 = np.einsum(
    "de,defg,fg->",
    bra_de, Z0Z1_np, ket_de
)

numpy_Z0I1 = np.einsum(
    "de,defg,fg->",
    bra_de, Z0I1_np, ket_de
)

print("NumPy  <Z0 Z1> =", numpy_Z0Z1)
print("NumPy  <Z0 I1> =", numpy_Z0I1)


# -----------------------------
# 5. Agreement check
# -----------------------------

print("Z0Z1 agrees:", np.allclose(numpy_Z0Z1.real, cudaq_Z0Z1))
print("Z0I1 agrees:", np.allclose(numpy_Z0I1.real, cudaq_Z0I1))



/usr/local/lib/python3.12/dist-packages/cupy/_environment.py:596: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda12x, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


CUDA-Q <Z0 Z1> = 0.9999999999999998
CUDA-Q <Z0 I1> = 0.0
NumPy  <Z0 Z1> = (0.9999999999999998+0j)
NumPy  <Z0 I1> = 0j
Z0Z1 agrees: True
Z0I1 agrees: True


Contracting this network would result in a scalar. Sampling follows a similar procedure where the marginal probability of the first qubit is computed, a measurement is sampled, that outcome is fixed in the network, and the procedure continues until all qubits are sampled. Unlike SV simulation where we start with a $2^N$ object, TN simulation starts with a collection of mostly rank-1, rank-2, and rank-4 tensors and contracts them down to a scalar. What determines the memory required for a TN computation is the contraction order, or contraction path, because that sets the size of the largest intermediate tensor.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 4:</h3>
    <p style="font-size: 16px; color: #333;">
Consider the following multiplication of $A_{ij} \times B_{jk} \times C_{kl}$, where the extent of each dimension is $(i: 10, j: 100, k: 10, l: 100)$. Demonstrate by computing the einsums that there is an optimal pathway to minimize the size of the intermediate tensor. Print the memory of the intermediate tensor and the time to perform the entire contraction in each case.
</div>


In [25]:
import numpy as np
import time

# EXERCISE 4
# ##TODO## Create tensors A, B, and C with the dimensions from the prompt.
# ##TODO## Contract them left-to-right and record the intermediate tensor size and runtime.
# ##TODO## Contract them right-to-left and record the intermediate tensor size and runtime.
# ##TODO## Print the intermediate memory usage for both contraction paths.
# ##TODO## Confirm that both paths produce the same final result.

import numpy as np
import time

# -----------------------------
# Create tensors
# -----------------------------

i, j, k, l = 10, 100, 10, 100

rng = np.random.default_rng(0)

A = rng.normal(size=(i, j))
B = rng.normal(size=(j, k))
C = rng.normal(size=(k, l))


# -----------------------------
# Helper for memory reporting
# -----------------------------

def memory_in_kb(array):
    return array.nbytes / 1024


# -----------------------------
# Path 1: Left-to-right
#
# A_ij B_jk C_kl
# First contract A and B:
#
# A_ij B_jk -> M_ik
#
# Then:
#
# M_ik C_kl -> D_il
# -----------------------------

start = time.perf_counter()

M_left = np.einsum("ij,jk->ik", A, B)
D_left = np.einsum("ik,kl->il", M_left, C)

end = time.perf_counter()

left_time = end - start
left_memory = memory_in_kb(M_left)


# -----------------------------
# Path 2: Right-to-left
#
# A_ij B_jk C_kl
# First contract B and C:
#
# B_jk C_kl -> M_jl
#
# Then:
#
# A_ij M_jl -> D_il
# -----------------------------

start = time.perf_counter()

M_right = np.einsum("jk,kl->jl", B, C)
D_right = np.einsum("ij,jl->il", A, M_right)

end = time.perf_counter()

right_time = end - start
right_memory = memory_in_kb(M_right)


# -----------------------------
# Check agreement
# -----------------------------

agree = np.allclose(D_left, D_right)


# -----------------------------
# Print results
# -----------------------------

print("Left-to-right path:")
print("  Intermediate equation: ij,jk->ik")
print("  Intermediate shape:", M_left.shape)
print(f"  Intermediate memory: {left_memory:.3f} KiB")
print(f"  Runtime: {left_time:.6e} seconds")

print()

print("Right-to-left path:")
print("  Intermediate equation: jk,kl->jl")
print("  Intermediate shape:", M_right.shape)
print(f"  Intermediate memory: {right_memory:.3f} KiB")
print(f"  Runtime: {right_time:.6e} seconds")

print()

print("Final output shape:", D_left.shape)
print("Both paths agree:", agree)


Left-to-right path:
  Intermediate equation: ij,jk->ik
  Intermediate shape: (10, 10)
  Intermediate memory: 0.781 KiB
  Runtime: 2.584383e-03 seconds

Right-to-left path:
  Intermediate equation: jk,kl->jl
  Intermediate shape: (100, 100)
  Intermediate memory: 78.125 KiB
  Runtime: 4.462880e-04 seconds

Final output shape: (10, 100)
Both paths agree: True


Clearly, even for a small problem, the path can matter significantly. This holds true for TN simulation too. If it is performed along a poor path, it is possible to run into the same memory requirements as SV simulation. In fact, deep circuits with high entanglement push TN simulation toward this limit and therefore become intractably slow and run into the same memory limits as SV. But if a circuit has low entanglement and is very shallow, it is possible to run an exact simulation of orders of magnitude more qubits using TN simulation.

Try the use cases in the example below to see this in action with CUDA-Q.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 4b:</h3>
    <p style="font-size: 16px; color: #333;">
Write a kernel that performs `qft_layers` of the QFT circuit. Time runs on the CUDA-Q SV and TN simulators. With deeper circuits, does the simulation time grow proportionally for each backend? In the following cell, try sampling a GHZ-state preparation kernel. How many qubits can you simulate in under 1 minute with SV on your device? How about TN?
</div>


In [ ]:
import numpy as np
import cudaq
import time

# EXERCISE 4B
# ##TODO## Write quantum_fourier_transform(qubits).
# ##TODO## Write qft(n, qft_layers) so it applies the QFT kernel multiple times.
# ##TODO## Time cudaq.sample(qft, 20, layer) on the 'nvidia' backend.
# ##TODO## Time the same circuit on the 'tensornet' backend.
# ##TODO## Compare how the runtime changes as the number of QFT layers increases.

@cudaq.kernel
def quantum_fourier_transform(q: cudaq.qview):
    n = len(q)

    # QFT body
    for target in range(n):
        h(q[target])

        for control in range(target + 1, n):
            angle = math.pi / (2.0 ** (control - target))
            r1.ctrl(angle, q[control], q[target])

    # Optional final bit-reversal swaps
    for i in range(n // 2):
        swap(q[i], q[n - i - 1])

@cudaq.kernel
def qft(n: int, qft_layers: int):
    q = cudaq.qvector(n)

    # Prepare a nontrivial computational basis input.
    # This avoids benchmarking only QFT(|00...0>).
    x(q[0])

    for _ in range(qft_layers):
        quantum_fourier_transform(q)

    mz(q)

def time_sample(target_name, n, qft_layers, shots=100):
    cudaq.set_target(target_name)

    # Warmup run. This helps reduce one-time compilation overhead.
    cudaq.sample(qft, n, qft_layers, shots_count=shots)

    start = time.perf_counter()
    counts = cudaq.sample(qft, n, qft_layers, shots_count=shots)
    stop = time.perf_counter()

    return stop - start, counts


layers_to_test = [1]#, 2, 4]#, 8, 16]
n = 2
shots = 100

sv_times = []
tn_times = []

for layer in layers_to_test:
    print(f"\nQFT layers = {layer}")

    try:
        sv_time, _ = time_sample("nvidia", n, layer, shots=shots)
        sv_times.append(sv_time)
        print(f"  nvidia   time: {sv_time:.6f} s")
    except Exception as e:
        sv_times.append(np.nan)
        print(f"  nvidia   failed: {e}")

    try:
        tn_time, _ = time_sample("tensornet", n, layer, shots=shots)
        tn_times.append(tn_time)
        print(f"  tensornet time: {tn_time:.6f} s")
    except Exception as e:
        tn_times.append(np.nan)
        print(f"  tensornet failed: {e}")


print("\nSummary")
print("layers | nvidia time (s) | tensornet time (s)")
print("-----------------------------------------------")

for layer, sv_t, tn_t in zip(layers_to_test, sv_times, tn_times):
    print(f"{layer:6d} | {sv_t:15.6f} | {tn_t:17.6f}")

/usr/local/lib/python3.12/dist-packages/cupy/_environment.py:596: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda12x, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


In [ ]:
import cudaq
import time

# EXERCISE 4B CONTINUED
# ##TODO## Write a GHZ-state preparation kernel.
# ##TODO## Time sampling the GHZ kernel on the 'nvidia' backend.
# ##TODO## Time sampling the GHZ kernel on the 'tensornet' backend.
# ##TODO## Increase the qubit count and compare how far each backend scales on your device.


In summary, TN simulations are great at extending beyond the memory limitations of SV simulations, especially when the circuit is shallow and local. For many circuits, the network description grows roughly with the number of qubits times the circuit depth, but the actual runtime is still dominated by the contraction path and the entanglement generated by the circuit. cuTensorNet is highly optimized to find good contraction pathways and run with peak performance on the GPU. TN simulations, if the circuit structure is right, provide most of the benefits of SV simulation aside from being able to save the resulting state.

A good recent example is found in ["Validating large-scale quantum machine learning: efficient simulation of quantum support vector machines using tensor networks"](https://iopscience.iop.org/article/10.1088/2632-2153/adb4ba), where a quantum SVM was constructed using 748 qubits to perform image classification.


## Matrix Product States

We have now covered simulators for deep circuits with few qubits and shallow circuits with many qubits. What do you do for the many cases that fall between these two regimes?

This is where matrix product state (MPS) simulators start to shine. Before exploring some examples, let's understand what an MPS is.

The main idea of an MPS state is to take a rank-$N$ tensor corresponding to the $2^N$ element state $\psi$ and decompose it into a special tensor network that looks like the following figure.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/simulation/images/mps_state.png?raw=1" alt="Matrix product state decomposition of an N-qubit state" width="700">

The vertical lines are called the physical dimension and represent a 1 or 0 for each state. If these are concrete values, then each node collapses from a rank-3 tensor to rank-2 tensors and hence represents the state as a product of matrices. At first glance, it is not obvious why this is advantageous over a standard TN approach, as we still produce the full state vector when we contract. However, MPS construction allows for systematic truncation of entanglement to shrink the state based on the singular value decomposition (SVD).

You may recall from linear algebra that an SVD is a tool to decompose any matrix into the following:

$$ A_{m \times n} = U_{m \times m} \Sigma_{m \times n} V_{n \times n} $$

Where $U$ and $V$ are unitary matrices whose columns and rows are populated by the eigenvectors of $AA^T$ or $A^TA$, respectively. $\Sigma$ is a diagonal matrix which contains the so-called singular values. The singular values are extremely useful because they quantify how information is stored in the data. This means you can truncate data that corresponds to small singular values with little impact on the information stored by the matrix.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 5:</h3>
    <p style="font-size: 16px; color: #333;">
Use the provided image `images/dog.png` and fix the following function to perform SVD on the image. This is a simple example of how image compression works. Given this matrix has about 400 singular values, how many are needed to get decent recovery of the image? How many are needed to achieve 90% of the information?
</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def analyze_image_svd(image_path, ranks=[1, 5, 10, 20, 50]):
    # 1. Load the image and convert to grayscale (L mode).
    img = Image.open(image_path).convert('L')
    img_matrix = np.array(img)

    # 2. Perform SVD.
    U, s, Vh = np.linalg.svd(img_matrix, full_matrices=False)
    spectrum_energy = s**2

    # 3. Setup visualization grid.
    n_ranks = len(ranks)
    fig, axes = plt.subplots(2, (n_ranks + 1) // 2 + 1, figsize=(18, 10))
    axes = axes.flatten()

    # Plot original image.
    axes[0].imshow(img_matrix, cmap='gray')
    axes[0].set_title("Original Image\n(Full Rank)")
    axes[0].set_xlabel(f"{img_matrix.nbytes:,} bytes")
    axes[0].axis('off')

    # 4. Reconstruct and plot truncated versions.
    for i, k in enumerate(ranks):
        # Truncation: keep only top-k singular values.
        Uk = U[:, :k]
        sk = s[:k]
        Vhk = Vh[:k, :]

        # Reconstruct: U * diag(s) * Vh.
        reconstructed = Uk @ np.diag(sk) @ Vhk
        compressed_bytes = Uk.nbytes + sk.nbytes + Vhk.nbytes

        # Display.
        ax_idx = i + 1
        axes[ax_idx].imshow(reconstructed, cmap='gray')
        axes[ax_idx].set_title(f"Rank k={k}")
        axes[ax_idx].set_xlabel(f"{compressed_bytes:,} bytes")
        axes[ax_idx].axis('off')

    # Hide any unused subplots.
    for j in range(len(ranks) + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

    # 5. Plot singular values (the spectrum).
    # ##TODO## Plot the singular values.
    # ##TODO## Plot the cumulative energy (information captured).
    # ##TODO## Mark the 90% information threshold and label the axes.
    # ##TODO## Display the spectrum figure.


# ##TODO## Call analyze_image_svd('images/dog.png').


The same principle that applies to image compression can extend to MPS construction of our quantum state. If we can leverage SVD cleverly, we can build a quantum state that captures the full wavefunction well without storing the entire state vector in memory. We can then use this state just like a TN simulator to compute observables or sample from.

The expression for an MPS is shown in the figure below. For a given $A$, the $i$ superscript is the physical dimension and would be set to 0 or 1 depending on the bitstring we want to sample. The $\alpha$'s are the bond dimensions between the matrices and are the part we truncate when performing the SVD.

The steps are as follows to convert a state vector $\psi$ into an MPS state. In practice, this would be done iteratively as each gate is applied in a circuit. Again, starting from the full state vector defeats the purpose of building a compact state we can store. However, to understand the process, it is pedagogically clearer to see how a full state is decomposed into an MPS.

We start by reshaping the state into a flattened $m \times n$ matrix where $m$ is the current bond dimension coming from the previous tensor times 2 to include the physical dimension of the current tensor. For the first tensor, $A_1$, we assume a dummy incoming bond dimension of 1. The $n$ dimension is the rest of the wavefunction, or $2^N/2$.

Next, SVD is performed on this matrix. We can then examine the singular values in $S$ and truncate the smallest ones so the retained rank does not exceed our chosen maximum bond dimension $\chi$. We also truncate $U$ and $V$ accordingly. In this first step, there will only be 2 singular values because we are performing SVD on a $2 \times 2^N/2$ matrix, so there is probably nothing to truncate yet.

$U$ is then saved as $A_1$ and we proceed to the next location $A_2$. Repeating the process, we now reshape $\psi$ into a matrix of size $(2 \times 2) \times 2^N/4$ because we now have an incoming bond dimension of 2 from the previous tensor $A_1$. This would correspond to $\alpha_1$ in the diagram above. Now, when we perform SVD, we get 4 singular values. If we set $\chi$ to 2, we truncate two of those values, and the same truncation is applied to $U$ and $V$.

The tensor $A_2$ now becomes this $U$ with $\alpha_2$ equal to 2 rather than 4 if we did not truncate. This is where we have trimmed data systematically while controlling how many parameters we store.

Moving on to $A_3$, we now have an incoming dimension of $\alpha_2 = 2$ rather than 4 without truncation. This process continues until the end. The result is a set of MPS tensors that can be contracted to return the approximate full state if truncation was used. In practice, an MPS simulator begins with the MPS structure and iteratively performs these SVDs as gates are applied, avoiding any need for the entire state vector.

Often, the MPS diagram will be depicted as in the figure below, where the thickness of the horizontal connections corresponds to the bond dimension.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/simulation/images/bond_dim.png?raw=1"  title="Landscape Image" width="300">


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 6:</h3>
    <p style="font-size: 16px; color: #333;">
Fix the code below to prepare an MPS state from an initial state vector and a $\chi$ value. Also complete the function that contracts the tensors to reconstruct the state. Analyze the impact of increasing the bond dimension on a 10 qubit GHZ state and a 10 qubit random state. Which one is more sensitive to truncation.  Test $\chi$ values of 8, 16, and 32.
</div>


In [ ]:
import numpy as np
from numpy import linalg as LA

# EXERCISE 6
def generate_random_state(N):
    # ##TODO## Generate a normalized random complex state of size 2**N.
    pass

def generate_ghz_state(N):
    # ##TODO## Generate the GHZ state (|00...0> + |11...1>) / sqrt(2).
    pass

def dense_to_mps(psi, N, max_bond):
    # ##TODO## Convert a dense state vector into an MPS using iterative SVD.
    # ##TODO## Truncate the singular spectrum at max_bond.
    pass

def mps_to_dense(mps_tensors):
    # ##TODO## Contract the MPS tensors back into a dense state vector.
    pass

def compute_fidelity(psi_original, psi_approx):
    # ##TODO## Compute the fidelity between the original and reconstructed states.
    pass

# ##TODO## Compare GHZ and random states for chi = 8, 16, and 32.
# ##TODO## Print or tabulate the resulting fidelities.


As you probably noticed from Exercise 6, the random state is far more sensitive to truncated bond dimension. The GHZ state, due to its highly structured entanglement, only requires a maximum bond dimension of 2. This demonstrates that MPS is best suited for circuits with low entanglement. Circuits with all-to-all connectivity can quickly suffer from MPS approximation, while systems like spin chains are well suited for MPS.

The key benefit of MPS simulation is that $\chi$ is tunable. It provides the flexibility to optimize the extent of approximation to match the entanglement needs of the circuit. Thus, MPS is a powerful middle ground between TN and SV simulators and is particularly useful for square-shaped circuits.

CUDA-Q allows users to run MPS simulations with the `tensornet-mps` backend, which handles the implementation details needed to execute quickly on a GPU. Users can specify the maximum bond dimension and other settings like which SVD algorithm to use. Note that just because MPS uses an approximation does not mean it is faster. In cases where SV simulation is available, it should be the first choice for performance, and MPS should be used when scaling beyond its reach.


## Pauli Propagation

Just because MPS and TN simulations can scale to larger qubit numbers, they are not necessarily fast enough for all situations. In the specific case where the goal is to calculate an expectation value of an operator, Pauli propagation can be used to obtain results much faster and at large scale.

Pauli propagation works unlike any of the other methods so far, which are all focused on building some exact or approximate representation of the state. Instead, Pauli propagation begins with the observable operator and propagates backwards through all of the gates in the circuit to determine the behavior of measuring such an operator.

A Clifford gate is a gate that maps Pauli strings to Pauli strings under conjugation. For example, $H Z H = X$, and CNOT maps a $Z$ on the target wire to $Z$ on both the control and target wires. This makes Clifford gates especially convenient for Pauli propagation.

Consider a simple two-qubit circuit where we want to measure $I_1 Z_2$ for a Bell-state circuit. First, we work from the end of the circuit and track the impact of the CNOT gate. A CNOT maps that operator to $Z_1 Z_2$. Finally, the $H$ gate acts on the operator corresponding to the first qubit and modifies the $Z$ to an $X$, resulting in a propagated Pauli operator of $X_1 Z_2$.

The expectation value (for the initial $\ket{00}$ state) $\bra{00} X_1 Z_2 \ket{00}$ is determined by summing the coefficients of the Pauli strings that only contain $I$ and $Z$ terms. In this example, there are none, so the result is 0. This makes sense because the operator $X_1 Z_2$ results in a term like $\bra{00}\ket{10} = 0$.

This first example makes it look easy. Why would we not do this all the time if it is much easier than preparing a state? The answer is that we have not yet encountered gates that cause branching. Consider measurement of $Z_1 Z_2$ in a two-qubit circuit which has a single $R_X(\pi/16)$ rotation gate on the first qubit. With Clifford gates, a single input Pauli results in a single output Pauli. Non-Clifford gates like $R_X$ take a single Pauli string to a linear combination of Pauli strings, a phenomenon called branching.

In this example, the $R_X$ gate takes $Z_1 Z_2$ to $\cos(\pi/16) Z_1 Z_2 + \sin(\pi/16) Y_1 Z_2 \approx 0.98 Z_1 Z_2 + 0.20 Y_1 Z_2$. Now, when we compute the expectation value with the initial $\ket{00}$ state, only the first term contributes so the expectation value is 0.98.

If you want to explore this more, try [this interactive Pauli propagation widget](https://nvidia.github.io/cuda-q-academic/interactive_widgets/pauli_prop_widget.html) where you can change the operators and see how it impacts the Pauli propagation of a few circuits.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 7:</h3>
    <p style="font-size: 16px; color: #333;">
Using CUDA-Q, confirm that the worked-out examples above, and some example selections from the widget, produce the correct expectation values.
</div>


In [ ]:
import numpy as np
import cudaq
from cudaq import spin

# EXERCISE 7
# ##TODO## Define a Bell-state kernel for the I_1 Z_2 example.
# ##TODO## Define an Rx(pi/16) example kernel for the Z_1 Z_2 example.
# ##TODO## Build the observables O1 = I_1 Z_2 and O2 = Z_1 Z_2.
# ##TODO## Use cudaq.observe to confirm the expectation values from the Pauli propagation discussion.
# ##TODO## If you add a widget later, test a few additional operator choices against the text discussion.


The primary limitation of Pauli propagation is branching, as it exponentially grows the number of terms that need to be tracked as the number of non-Clifford gates grows. This is often handled by truncation. Consider some string like $0.58 P_1 + 0.79 P_2 + 0.01 P_3 + 0.20 P_4$. One might perform Pauli propagation with some preset threshold like 0.02 and then drop terms that have coefficients below this, in this case $P_3$. The remaining terms would then be renormalized and tracked without the need to follow any of the results that would come from the $P_3$ branch. Note that it may be tempting to drop terms that currently do not contribute to the expectation value, but the remaining gates may transform these terms into significant contributions, so there is no way to know a priori which terms have no impact.

Truncation based on a minimum threshold makes Pauli propagation scalable, but it also introduces error with every truncation step, and those errors can accumulate and corrupt results. Thus, Pauli propagation is a fantastic tool (without truncation) for large circuits with relatively few non-Clifford gates. It is also great (with truncation) for evaluating cheap expectation values for large circuits, but great care must be taken to ensure truncation does not have too great an impact on the end result.

Another important limitation is that Pauli propagation tracks observables rather than amplitudes, so it does not directly produce bitstring measurement samples from the final circuit. For example, it can help optimize the QAOA cost Hamiltonian, but if you ultimately want a candidate max-cut bitstring from the optimized circuit, you would still turn to another simulator such as TN or MPS to sample that circuit.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 8:</h3>
    <p style="font-size: 16px; color: #333;">

Options here: 1. Skip this exercise for now. 2. Ideally, wait for CUDA-Q to expose PP directly and then compare MPS and PP on the same expectation-value task. 3. As an advanced extension, use cuPP and cuTensorNet directly in a standalone script.
</div>


## Stabilizer Simulation

The final simulation technique we discuss here is stabilizer simulation. Stabilizer simulation works similarly to Pauli propagation, but only for Clifford circuits. Stabilizers (covered in depth in QEC 101 Lab 2) are special operators that stabilize a state. That is, if $\ket{\psi}$ lies in the stabilized subspace, a stabilizer $S$ acts on it such that $S\ket{\psi} = \ket{\psi}$. In that case, the stabilizer returns the original state.

Each Clifford circuit can be associated with a stabilizer group which essentially describes the constraints required to build that circuit. A simulation usually begins with the stabilizer $Z_1 Z_2 \dots Z_n$, which stabilizes $\ket{0 \dots 0}$. Then, just like for Pauli propagation, this stabilizer is tracked as all of the gates in the circuit are applied. Because none of them are non-Clifford, branching does not occur so tens of thousands of qubits can be simulated.

Stabilizer simulation is not suited for general algorithm testing because any useful quantum algorithm will require non-Clifford gates; otherwise it is efficiently classically simulable per the Gottesman-Knill theorem. However, it is a powerful technique for QEC researchers.

For example, if you developed a new QEC code, you may want to determine its threshold, the physical error rate for which adding more physical qubits lowers the logical error rate rather than makes things worse. You could run large-scale noisy stabilizer simulations with Clifford circuits and Pauli noise models and numerically evaluate the threshold. Similarly, you can use stabilizer simulation to test protocols for techniques like magic state distillation.

Stabilizer simulators are also workhorses for generating synthetic data that can be used for testing new decoders and training QEC-related AI models.

Extracting expectation values and samples from a stabilizer simulation is somewhat involved and beyond the scope of this lesson. Tools like cuStabilizer (GPU-accelerated stabilizer simulation) and CUDA-Q's STIM backend handle this for you.

For a detailed example where stabilizer simulation is useful for QEC, check out the [QEC 101 notebook on topological codes](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/06_QEC_Topological_Codes.ipynb), where a stabilizer simulator is used to prepare an example of the distance-3 surface code.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
    <h3 style="color: #76b900; margin-top: 0; margin-bottom: 10px;">Exercise 9:</h3>
    <p style="font-size: 16px; color: #333;">

The CUDA-Q STIM backend is fantastic for simulating QEC experiments. Try running the code below which performs a memory experiment with the surface code. If you are not familiar with QEC or memory experiments, check out the CUDA-Q Academic QEC 101 course lessons 1 and 2. If you are familiar with QEC, study the workflow to understand how the STIM backend is used here.

Each sample produces a list of measurements of the ancilla qubits (stored in `syndrome`) and the data qubits (stored as `data`). Add a print statement to see how many qubits are simulated for surface codes of distance 3, 5, 7, and 9.

</div>


In [ ]:
import numpy as np
import cudaq
import cudaq_qec as qec

# EXERCISE 9
# ##TODO## Select the STIM backend and construct a surface-code memory experiment.
# ##TODO## Build the detector error model for the chosen code distance and noise model.
# ##TODO## Sample the noisy memory circuit and inspect the syndrome and data arrays.
# ##TODO## Print how many ancilla and data qubits are simulated for distances 3, 5, 7, and 9.
# ##TODO## Decode the syndromes and compare logical errors before and after decoding.


## Summary

Hopefully it is now clear that there are many different tools available for quantum algorithm simulation depending on the situation. You should now have an understanding of the basics of how each simulator works, what its strengths and weaknesses are, and what some common use cases are.

For most situations, you will probably still use SV, TN, and MPS simulations. The chart below gives a reasonable qualitative representation of the three regimes where each is most useful.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/simulation/images/backend_chart.png?raw=1" alt="Qualitative comparison chart for SV, TN, and MPS simulation regimes" width="700">

In cases where cheap expectation values are required, Pauli propagation is a tool to consider. When you need large-circuit sampling rather than just observables, TN or MPS may still be the better choice. For specialized cases, often related to QEC, stabilizer simulation is the best choice.

Each of these simulation methods can benefit from GPU acceleration, often with implementations that are far from trivial. The NVIDIA CUDA-Q platform and underlying cuQuantum libraries make it easy to benefit from whatever sort of accelerated simulation you need. The [CUDA-Q applications page](https://nvidia.github.io/cuda-quantum/latest/using/applications.html) and the other [CUDA-Q Academic learning paths](https://github.com/NVIDIA/cuda-q-academic) have many examples demonstrating cases where these simulation tools can be used for a diverse range of applications.
